<a href="https://colab.research.google.com/github/Sudeepcpc/AWS-Bedrock/blob/main/transformers_doc/en/pytorch/language_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Transformers installation
! pip install transformers datasets evaluate accelerate
# To install from source instead of the last release, comment the command above and uncomment the following one.
# ! pip install git+https://github.com/huggingface/transformers.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


# Causal language modeling

There are two types of language modeling, causal and masked. This guide illustrates causal language modeling.
Causal language models are frequently used for text generation. You can use these models for creative applications like
choosing your own text adventure or an intelligent coding assistant like Copilot or CodeParrot.

In [ ]:
#@title
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/Vpjb1lu0MDk?rel=0&amp;controls=0&amp;showinfo=0" frameborder="0" allowfullscreen></iframe>')

Causal language modeling predicts the next token in a sequence of tokens, and the model can only attend to tokens on
the left. This means the model cannot see future tokens. GPT-2 is an example of a causal language model.

This guide will show you how to:

1. Finetune [DistilGPT2](https://huggingface.co/distilbert/distilgpt2) on the [r/askscience](https://www.reddit.com/r/askscience/) subset of the [ELI5](https://huggingface.co/datasets/dany0407/eli5_category) dataset.
2. Use your finetuned model for inference.

<Tip>

To see all architectures and checkpoints compatible with this task, we recommend checking the [task-page](https://huggingface.co/tasks/text-generation)

</Tip>

Before you begin, make sure you have all the necessary libraries installed:

```bash
pip install transformers datasets evaluate
```

We encourage you to log in to your Hugging Face account so you can upload and share your model with the community. When prompted, enter your token to log in:

In [1]:
from huggingface_hub import notebook_login

notebook_login()

## Load ELI5 dataset

Start by loading the first 5000 examples from the [ELI5-Category](https://huggingface.co/datasets/dany0407/eli5_category) dataset with the 🤗 Datasets library. This'll give you a chance to experiment and make sure everything works before spending more time training on the full dataset.

In [2]:
from datasets import load_dataset

eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")

README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 98.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation1-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.92MB            

data/validation1-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/91772 [00:00<?, ? examples/s]

Generating validation1 split:   0%|          | 0/5446 [00:00<?, ? examples/s]

Generating validation2 split:   0%|          | 0/2375 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5411 [00:00<?, ? examples/s]

Split the dataset's `train` split into a train and test set with the [train_test_split](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.train_test_split) method:

In [3]:
eli5 = eli5.train_test_split(test_size=0.2)

Then take a look at an example:

In [4]:
eli5["train"][0]

{'q_id': '5piml3',
 'title': 'why acceleration is constant in a vacuum, and an airplane would fall at the same velocity as a tennis ball?',
 'selftext': '',
 'category': 'Physics',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dcrfibj', 'dcrijlf'],
  'text': ['Because in a vacuum a moving object has no resistance acting on it. Clarify the second part of the question please.',
   "The force of gravity acting on an object is proportional to its mass. Mass times gravity = weight. Acceleration is force divided by mass. For example, if I push on a small rock, it accelerates away from me pretty quickly while if I push on a boulder, it takes a long time to start moving with any speed. (Ignoring friction, of course; pretend both of them are on frictionless rails.) Note that weight is just a kind of force. So if weight = gravity * mass and acceleration down = weight ÷ mass then acceleration down = gravity * mass ÷ mass and the mass cancels out leaving acceleration down = gravity All

While this may look like a lot, you're only really interested in the `text` field. What's cool about language modeling
tasks is you don't need labels (also known as an unsupervised task) because the next word *is* the label.

## Preprocess

In [ ]:
#@title
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/ma1TrR7gE7I?rel=0&amp;controls=0&amp;showinfo=0" frameborder="0" allowfullscreen></iframe>')

The next step is to load a DistilGPT2 tokenizer to process the `text` subfield:

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

You'll notice from the example above, the `text` field is actually nested inside `answers`. This means you'll need to
extract the `text` subfield from its nested structure with the [`flatten`](https://huggingface.co/docs/datasets/process#flatten) method:

In [6]:
eli5 = eli5.flatten()
eli5["train"][0]

{'q_id': '5piml3',
 'title': 'why acceleration is constant in a vacuum, and an airplane would fall at the same velocity as a tennis ball?',
 'selftext': '',
 'category': 'Physics',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dcrfibj', 'dcrijlf'],
 'answers.text': ['Because in a vacuum a moving object has no resistance acting on it. Clarify the second part of the question please.',
  "The force of gravity acting on an object is proportional to its mass. Mass times gravity = weight. Acceleration is force divided by mass. For example, if I push on a small rock, it accelerates away from me pretty quickly while if I push on a boulder, it takes a long time to start moving with any speed. (Ignoring friction, of course; pretend both of them are on frictionless rails.) Note that weight is just a kind of force. So if weight = gravity * mass and acceleration down = weight ÷ mass then acceleration down = gravity * mass ÷ mass and the mass cancels out leaving acceleration down = gravity A

Each subfield is now a separate column as indicated by the `answers` prefix, and the `text` field is a list now. Instead
of tokenizing each sentence separately, convert the list to a string so you can jointly tokenize them.

Here is a first preprocessing function to join the list of strings for each example and tokenize the result:

In [7]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

To apply this preprocessing function over the entire dataset, use the 🤗 Datasets [map](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.map) method. You can speed up the `map` function by setting `batched=True` to process multiple elements of the dataset at once, and increasing the number of processes with `num_proc`. Remove any columns you don't need:

In [8]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2815 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1823 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1123 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2153 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2066 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1305 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1980 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (8074 > 1024). Running this sequence through the model will result in indexing errors


This dataset contains the token sequences, but some of these are longer than the maximum input length for the model.

You can now use a second preprocessing function to

- concatenate all the sequences
- split the concatenated sequences into shorter chunks defined by `block_size`, which should be both shorter than the maximum input length and short enough for your GPU RAM.

In [9]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

Apply the `group_texts` function over the entire dataset:

In [10]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Now create a batch of examples using [DataCollatorForLanguageModeling](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorForLanguageModeling). It's more efficient to *dynamically pad* the
sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

Use the end-of-sequence token as the padding token and set `mlm=False`. This will use the inputs as labels shifted to the right by one element:

In [11]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Train

<Tip>

If you aren't familiar with finetuning a model with the [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer), take a look at the [basic tutorial](https://huggingface.co/docs/transformers/main/en/tasks/../training)!

</Tip>

You're ready to start training your model now! Load DistilGPT2 with [AutoModelForCausalLM](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModelForCausalLM):

In [12]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

At this point, only three steps remain:

1. Define your training hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments). The only required parameter is `output_dir` which specifies where to save your model. You'll push this model to the Hub by setting `push_to_hub=True` (you need to be signed in to Hugging Face to upload your model).
2. Pass the training arguments to [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) along with the model, datasets, and data collator.
3. Call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

In [13]:
training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.922134,3.805601
2,3.822813,3.795882
3,3.776095,3.794120


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3933, training_loss=3.846170859454456, metrics={'train_runtime': 1048.6467, 'train_samples_per_second': 29.993, 'train_steps_per_second': 3.751, 'total_flos': 1027288174952448.0, 'train_loss': 3.846170859454456, 'epoch': 3.0})

Once training is completed, use the [evaluate()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.evaluate) method to evaluate your model and get its perplexity:

In [14]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
3.776095,3.794120,3


Perplexity: 44.44


Then share your model to the Hub with the [push_to_hub()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.push_to_hub) method so everyone can use your model:

In [15]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/sudeep1947/my_awesome_eli5_clm-model/commit/ac3e9304ccd1e8ed36d0e594bead2281a7178ed8', commit_message='End of training', commit_description='', oid='ac3e9304ccd1e8ed36d0e594bead2281a7178ed8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sudeep1947/my_awesome_eli5_clm-model', endpoint='https://huggingface.co', repo_type='model', repo_id='sudeep1947/my_awesome_eli5_clm-model'), pr_revision=None, pr_num=None)

<Tip>

For a more in-depth example of how to finetune a model for causal language modeling, take a look at the corresponding
[PyTorch notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/language_modeling.ipynb).

</Tip>

## Inference

Great, now that you've finetuned a model, you can use it for inference!

Come up with a prompt you'd like to generate text from:

In [16]:
prompt = "Somatic hypermutation allows the immune system to"

The simplest way to try out your finetuned model for inference is to use it in a [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline). Instantiate a `pipeline` for text generation with your model, and pass your text to it:

In [17]:
from transformers import pipeline

generator = pipeline("text-generation", model="sudeep1947/my_awesome_eli5_clm-model")
generator(prompt)

config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  328MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/326 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'Somatic hypermutation allows the immune system to control a person\'s immune response to an infection. The immune system can also target animals for their immune system to be attacked by animals. This is why some people have a negative immune response and want to eat more meat. In the UK it is the UK where it is illegal for a person to eat animals without food. However, in the US it is illegal for a person to eat a meat without food. In other words, you can eat meat without food in the UK. The UK is a country where it is illegal for a person to eat meat without food. Since the UK is a country with a certain number of different laws in place, there are usually laws in place to allow "unhealthy" meat to be eaten, and the UK is an example. For example, if you are planning to cook a meat without food in the UK (i.e. eating a meat that was not unhealthy) you can, in a state where you are allowed to eat it without food in the UK, but your state does have laws in place to

Tokenize the text and return the `input_ids` as PyTorch tensors:

In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sudeep1947/my_awesome_eli5_clm-model")
inputs = tokenizer(prompt, return_tensors="pt").input_ids

Use the [generate()](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation#transformers.GenerationMixin.generate) method to generate text.
For more details about the different text generation strategies and parameters for controlling generation, check out the [Text generation strategies](https://huggingface.co/docs/transformers/main/en/tasks/../generation_strategies) page.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("username/my_awesome_eli5_clm-model")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)

Decode the generated token ids back into text:

In [ ]:
tokenizer.batch_decode(outputs, skip_special_tokens=True)

["Somatic hypermutation allows the immune system to react to drugs with the ability to adapt to a different environmental situation. In other words, a system of 'hypermutation' can help the immune system to adapt to a different environmental situation or in some cases even a single life. In contrast, researchers at the University of Massachusetts-Boston have found that 'hypermutation' is much stronger in mice than in humans but can be found in humans, and that it's not completely unknown to the immune system. A study on how the immune system"]

## Comprehensive Code and Output Explanation

### 1. Installation of Dependencies
```python
! pip install transformers datasets evaluate accelerate
```
* **Line-by-Line:**
  * `! pip install`: Runs a shell command in the notebook environments to install python packages.
  * `transformers datasets evaluate accelerate`: Installs the Hugging Face ecosystem libraries for building, training, and evaluating NLP models.
* **Output:** Shows that the packages (like `evaluate-0.4.6`) were successfully installed, while most others were already satisfied in the environment.

---

### 2. Loading the Dataset
```python
from datasets import load_dataset
eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")
```
* **Line-by-Line:**
  * `from datasets import load_dataset`: Imports the function used to load Hugging Face Hub datasets.
  * `eli5 = load_dataset(...)`: Downloads and loads the first 5,000 training examples from the `dany0407/eli5_category` dataset (a simplified version of Reddit's *Explain Like I'm Five* subset).

---

### 3. Splitting and Viewing Data
```python
eli5 = eli5.train_test_split(test_size=0.2)
eli5["train"][0]
```
* **Line-by-Line:**
  * `eli5.train_test_split(test_size=0.2)`: Partitions the 5,000 samples into an 80% training set (4,000 samples) and a 20% test set (1,000 samples).
  * `eli5["train"][0]`: Retrieves the first dictionary object from the newly created training split to inspect its structure.
* **Output:** A nested dictionary containing key-value pairs representing the question ID (`q_id`), `title`, `category`, and nested answers containing a list of `text` solutions and their respective scores.

---

### 4. Flattening the Dataset
```python
eli5 = eli5.flatten()
eli5["train"][0]
```
* **Line-by-Line:**
  * `eli5 = eli5.flatten()`: Flattens nested dictionary structures. For example, `answers` containing a dictionary list gets converted into independent columns like `answers.text` and `answers.score`.
* **Output:** The dictionary keys change (e.g., `'answers': {'text': [...]}` becomes `'answers.text': [...]`).

---

### 5. Tokenization Preprocessing
```python
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])
```
* **Line-by-Line:**
  * `def preprocess_function(examples)`: Declares a function to batch-process row slices.
  * `" ".join(x) for x in ...`: Joins all the lists of text answers for a given question into a single continuous block of string text.
  * `tokenizer(...)`: Converts the raw text strings into numeric token IDs that the model can understand.

```python
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)
```
* **Line-by-Line:**
  * `eli5.map(...)`: Applies the preprocessing function across the entire train and test dataset.
  * `batched=True`: Processes groups of samples concurrently.
  * `num_proc=4`: Uses 4 CPU cores parallelly to quicken preprocessing.
  * `remove_columns=...`: Discards the non-numeric metadata columns since the model only requires `input_ids` and `attention_mask`.

---

### 6. Grouping and Chunking Text
```python
block_size = 128

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result
```
* **Line-by-Line:**
  * `concatenated_examples = ...`: Merges all dataset token lists into a single continuous stream of integers.
  * `total_length = ...`: Finds the maximum length that is cleanly divisible by `block_size` (128) and drops the remainder fraction.
  * `[t[i : i + block_size] for ...]`: Splits the mega-list of integers into equal-sized chunks of exactly 128 tokens.
  * `result["labels"] = result["input_ids"].copy()`: Duplicate the tokens into a `'labels'` column. In Causal Language Modeling, labels are shifted internally by the loss function so the model learns to predict the next token.

---

### 7. Data Collator and Model Training
```python
from transformers import DataCollatorForLanguageModeling
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
```
* **Line-by-Line:**
  * `tokenizer.pad_token = tokenizer.eos_token`: Sets the End-Of-Sequence token as the padding token since GPT-2 lacks an explicit default pad token.
  * `mlm=False`: Tells the data collator we are performing Causal Language Modeling (left-to-right generation) rather than Masked Language Modeling (filling in the blanks).

```python
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()
```
* **Line-by-Line:**
  * `Trainer(...)`: Standardizes the pipeline loop, binding the core model, training arguments, data structures, and collators together.
  * `trainer.train()`: Performs the forward passes, loss computations, backward passes, and gradient updates.

---

### 8. Evaluation
```python
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
```
* **Line-by-Line:**
  * `trainer.evaluate()`: Computes loss metrics on the held-out test dataset.
  * `math.exp(eval_results['eval_loss'])`: Converts cross-entropy loss to Perplexity ($e^{\text{loss}}$) - a traditional benchmark measuring how confident a language model is on test text (lower is better).
* **Output:** `Perplexity: 49.61` (shows the model is decently suited to conversational generation tasks).

---

### 9. Generating New Text (Inference)
```python
generator = pipeline("text-generation", model="username/my_awesome_eli5_clm-model")
generator(prompt)
```
* **Line-by-Line:**
  * `pipeline("text-generation", ...)`: Loads a high-level API pipeline containing both the model weights and tokenizer.
  * `generator(prompt)`: Takes your custom text starter prompt and auto-completes the sentence until hitting predefined generation length limits.
* **Output:** Generates a structured paragraph starting with your original text: *"Somatic hypermutation allows the immune system to..."*